# **Project 4: Agentic RAG with LangGraph**

Patterns from the course used here:

| Course notebook | Used for |
|---|---|
| 1 first graph / 2 Pydantic | state schema and structured output |
| 4 prompts & chains | every LLM step is `prompt \| llm` |
| 8 router | `check_scope` and the routing functions (conditional edges with a mapping dict) |
| 10 generator-evaluator | generate -> check grounding -> regenerate (with a counter) |
| 11 memory | `InMemorySaver` with a `thread_id` |

Flow: **check_scope -> retrieve -> grade_documents -> (rewrite once | abstain) -> generate -> check_grounding -> (regenerate once | abstain | answer)**

In [15]:
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
import os

load_dotenv()

if os.environ['OPENROUTER_API_KEY']:
    print("OpenRouter API Key is set")
else:
    raise ValueError("OpenRouter API Key is not set")

OpenRouter API Key is set


### **Models**
Use a **different** model for the checking steps (scope, grading, rewriting, grounding) than for generating the answer, so one model does not mark its own homework.
If a model name is not available on your OpenRouter account, change it.

In [16]:
llm = ChatOpenRouter(model='gpt-5-nano')                          # generates the answer
checker_llm = ChatOpenRouter(model='google/gemini-2.5-flash-lite')   # scope, grader, rewrite, grounding (change if unavailable)
judge_llm = ChatOpenRouter(model='anthropic/claude-haiku-4.5')       # used only in the evaluation at the end (change if unavailable)

In [17]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5", encode_kwargs={"normalize_embeddings": True})

vectorstore = Chroma(
    collection_name="employment_act",
    embedding_function=embeddings,
    persist_directory="../chroma_db",
)

print(vectorstore._collection.count(), "chunks in the store")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1728.70it/s]


168 chunks in the store


In [18]:
import re

TOP_K = 5
INSUFFICIENT = "INSUFFICIENT_EVIDENCE"

def retrieve_docs(query, k=TOP_K):
    # search the vector store and keep only what we need from each result
    results = vectorstore.similarity_search(query, k=k)
    return [{"content": r.page_content,
             "section_no": r.metadata["section_no"],
             "section_title": r.metadata["section_title"]} for r in results]

def format_context(docs):
    # one block per section, labelled so the LLM can cite it, e.g. [s.60A]
    return "\n\n".join(f"[s.{d['section_no']}] {d['section_title']}\n{d['content']}" for d in docs)

def get_citations(answer):
    # Captures "s.60A" even if the LLM wrote "[s.60A(2)]" or "[s.60A(3)(a)]"
    matches = re.findall(r'\[s\.([0-9]+[A-Za-z]*)', answer)
    return sorted({f"s.{m}" for m in matches})

### **Step 1: Structured outputs (Pydantic, like notebook 8)**
`Field(description=...)` tells the LLM what each field means.

In [19]:
from pydantic import BaseModel, Field
from typing import Literal, List

class scope_schema(BaseModel):
    category: Literal['in_scope', 'sabah_sarawak', 'out_of_scope'] = Field(..., description="Is the question in scope for the Employment Act 1955?")

class grade_schema(BaseModel):
    relevant_ids: List[int] = Field(..., description="ids of the sections that are useful for answering the question; empty list if none")

class grounding_schema(BaseModel):
    supported: bool = Field(..., description="True only if every claim in the answer is supported by the context")
    unsupported_claims: List[str] = Field(..., description="the claims that are not supported; empty list if none")

### **Step 2: Prompts and chains (like notebook 4)**

In [20]:
from langchain_core.prompts import ChatPromptTemplate

scope_prompt = ChatPromptTemplate.from_messages([
    ("system", "You classify questions for a Malaysian employment-law assistant. "
               "The knowledge base covers ONLY the Employment Act 1955 (Peninsular Malaysia and Labuan). "
               "in_scope: about employment rights or duties that this Act could govern. "
               "sabah_sarawak: asks specifically about Sabah or Sarawak (they have separate labour ordinances). "
               "out_of_scope: anything else, such as tax, immigration, other countries or unrelated topics."),
    ("human", "{question}"),
])

grade_prompt = ChatPromptTemplate.from_messages([
    ("system", "You grade retrieved legal sections for relevance to a question. "
               "Return the ids of the sections that contain information useful for answering it. "
               "Return an empty list if none are useful."),
    ("human", "Question: {question}\n\nSections:\n{numbered_docs}"),
])

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the user's question as a short search query using the wording of statutes, "
               "for example: hours of work, termination notice, maternity leave entitlement. "
               "Keep it under 20 words. Return ONLY the query."),
    ("human", "{question}"),
])

grounding_prompt = ChatPromptTemplate.from_messages([
    ("system", "You check an answer against its context. supported is true only if EVERY factual claim in the answer "
               "is supported by the context and every citation label points to a context item that really supports the claim. "
               "Otherwise list the unsupported claims."),
    ("human", "Context:\n{context}\n\nAnswer:\n{answer}"),
])

scope_chain = scope_prompt | checker_llm.with_structured_output(scope_schema)
grade_chain = grade_prompt | checker_llm.with_structured_output(grade_schema)
rewrite_chain = rewrite_prompt | checker_llm
grounding_chain = grounding_prompt | checker_llm.with_structured_output(grounding_schema)

In [21]:
from langchain_core.prompts import ChatPromptTemplate

generate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You explain the Malaysian Employment Act 1955. Answer ONLY using the context. "
               "Cite every claim with its label in square brackets, exactly as given, for example [s.60A]. "
               "Do not cite anything that is not in the context. "
               "If the context does not contain enough information, reply exactly: INSUFFICIENT_EVIDENCE. "
               "This is general information, not legal advice."),
    ("human", "Context:\n{context}\n\nQuestion: {question}\n{feedback}"),
])

generate_chain = generate_prompt | llm

### **Step 3: The state schema**
Rule: put something in the state only if a **later node, a routing decision, or the app** needs it.

In [22]:
from typing import TypedDict

class graph_schema(TypedDict):
    question: str          # what the user asked
    search_query: str      # what we search with (changes when the query is rewritten)
    documents: List[dict]  # the sections we are using
    rewrite_count: int     # how many rewrites we have used
    regen_count: int       # how many regenerations we have used
    answer: str
    feedback: str          # unsupported claims, sent back to the generator
    status: str            # answered / abstained
    abstain_reason: str    # sabah_sarawak / out_of_scope / no_evidence / unsupported
    path: List[str]        # nodes visited (useful to debug and to show in the app)

MAX_REWRITES = 1
MAX_REGENS = 1

def initial_state(question):
    # every question starts with a fresh state (the checkpointer would otherwise keep old counters)
    return {"question": question, "search_query": question, "documents": [],
            "rewrite_count": 0, "regen_count": 0, "answer": "", "feedback": "",
            "status": "", "abstain_reason": "", "path": []}

### **Step 4: The node functions**
Like the router notebook, each node returns **only the fields it changes**.

In [23]:
def check_scope(state: graph_schema) -> graph_schema:
    result = scope_chain.invoke({"question": state['question']})

    if result.category != 'in_scope':
        return {'status': 'abstained', 'abstain_reason': result.category,
                'path': state['path'] + ['check_scope(out)']}

    return {'path': state['path'] + ['check_scope']}


def retrieve(state: graph_schema) -> graph_schema:
    docs = retrieve_docs(state['search_query'])
    return {'documents': docs, 'path': state['path'] + ['retrieve']}


def grade_documents(state: graph_schema) -> graph_schema:
    docs = state['documents']

    numbered_docs = "\n\n".join(
        f"id={i}\n[s.{d['section_no']}] {d['section_title']}\n{d['content'][:800]}"
        for i, d in enumerate(docs))

    result = grade_chain.invoke({"question": state['question'], "numbered_docs": numbered_docs})

    # keep only the sections the grader said are relevant
    kept = [docs[i] for i in result.relevant_ids if 0 <= i < len(docs)]

    return {'documents': kept, 'path': state['path'] + [f"grade({len(kept)}/{len(docs)})"]}


def rewrite_query(state: graph_schema) -> graph_schema:
    new_query = rewrite_chain.invoke({"question": state['question']}).content.strip()
    return {'search_query': new_query, 'rewrite_count': state['rewrite_count'] + 1,
            'path': state['path'] + ['rewrite_query']}


def generate(state: graph_schema) -> graph_schema:
    feedback_text = ""
    if state['feedback']:
        feedback_text = f"A previous draft had unsupported claims: {state['feedback']}. Fix this."

    answer = generate_chain.invoke({"context": format_context(state['documents']),
                                    "question": state['question'],
                                    "feedback": feedback_text}).content.strip()

    if INSUFFICIENT in answer:
        return {'answer': answer, 'status': 'abstained', 'abstain_reason': 'no_evidence',
                'path': state['path'] + ['generate(insufficient)']}

    return {'answer': answer, 'path': state['path'] + ['generate']}


def check_grounding(state: graph_schema) -> graph_schema:
    result = grounding_chain.invoke({"context": format_context(state['documents']), "answer": state['answer']})

    if result.supported:
        return {'status': 'answered', 'path': state['path'] + ['grounding(ok)']}

    new_count = state['regen_count'] + 1

    if new_count > MAX_REGENS:       # already regenerated once: give up
        return {'status': 'abstained', 'abstain_reason': 'unsupported', 'regen_count': new_count,
                'path': state['path'] + ['grounding(fail)']}

    return {'feedback': "; ".join(result.unsupported_claims), 'regen_count': new_count,
            'path': state['path'] + ['grounding(fail)']}


ABSTAIN_MESSAGES = {
    "sabah_sarawak": "Sabah and Sarawak have their own labour ordinances, which this system does not cover.",
    "out_of_scope": "This question is outside the scope of the Employment Act 1955 knowledge base.",
    "no_evidence": "I could not find enough reliable evidence in the Employment Act 1955 to answer this.",
    "unsupported": "I could not produce an answer that was fully supported by the source text.",
}

def abstain(state: graph_schema) -> graph_schema:
    message = ABSTAIN_MESSAGES.get(state['abstain_reason'], ABSTAIN_MESSAGES["no_evidence"])
    message += " Please check with the Labour Department (JTKSM) or a qualified professional."
    return {'answer': message, 'status': 'abstained', 'path': state['path'] + ['abstain']}

### **Step 5: The routing functions (conditional edges, like notebook 8)**
Each one returns the **name of the next step**. The mapping dict in `add_conditional_edges` turns that name into a node.

In [24]:
def route_after_scope(state: graph_schema) -> str:
    if state['status'] == 'abstained':
        return 'abstain'
    return 'retrieve'


def route_after_grade(state: graph_schema) -> str:
    if state['documents']:                              # something relevant was found
        return 'generate'
    if state['rewrite_count'] < MAX_REWRITES:           # a retry is still left
        return 'rewrite_query'
    return 'abstain'


def route_after_generate(state: graph_schema) -> str:
    if state['status'] == 'abstained':                  # the model said INSUFFICIENT_EVIDENCE
        return 'abstain'
    return 'check_grounding'


def route_after_grounding(state: graph_schema) -> str:
    if state['status'] == 'answered':
        return 'end'
    if state['status'] == 'abstained':
        return 'abstain'
    return 'generate'                                   # regenerate once, with feedback

### **Step 6: Build and compile the graph**
`InMemorySaver` and `thread_id` work like notebook 11.

In [25]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(graph_schema)

# Adding Nodes
graph.add_node("check_scope", check_scope)
graph.add_node("retrieve", retrieve)
graph.add_node("grade_documents", grade_documents)
graph.add_node("rewrite_query", rewrite_query)
graph.add_node("generate", generate)
graph.add_node("check_grounding", check_grounding)
graph.add_node("abstain", abstain)

# Adding Edges
graph.add_edge(START, "check_scope")

graph.add_conditional_edges("check_scope", route_after_scope, {
    "retrieve": "retrieve",
    "abstain": "abstain",
})

graph.add_edge("retrieve", "grade_documents")

graph.add_conditional_edges("grade_documents", route_after_grade, {
    "generate": "generate",
    "rewrite_query": "rewrite_query",
    "abstain": "abstain",
})

graph.add_edge("rewrite_query", "retrieve")      # the retry loop

graph.add_conditional_edges("generate", route_after_generate, {
    "check_grounding": "check_grounding",
    "abstain": "abstain",
})

graph.add_conditional_edges("check_grounding", route_after_grounding, {
    "end": END,
    "abstain": "abstain",
    "generate": "generate",                       # the regenerate loop
})

graph.add_edge("abstain", END)

In [26]:
from IPython.display import Image, display
from langgraph.checkpoint.memory import InMemorySaver

checkpoint = InMemorySaver()

agent = graph.compile(checkpointer=checkpoint)

# You could see the errors with the below command
Image(agent.get_graph().draw_mermaid_png())

# You can use the below command to see the code of graph without errors
# then copy it to paste in mermaid.live website
print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	check_scope(check_scope)
	retrieve(retrieve)
	grade_documents(grade_documents)
	rewrite_query(rewrite_query)
	generate(generate)
	check_grounding(check_grounding)
	abstain(abstain)
	__end__([<p>__end__</p>]):::last
	__start__ --> check_scope;
	check_grounding -. &nbsp;end&nbsp; .-> __end__;
	check_grounding -.-> abstain;
	check_grounding -.-> generate;
	check_scope -.-> abstain;
	check_scope -.-> retrieve;
	generate -.-> abstain;
	generate -.-> check_grounding;
	grade_documents -.-> abstain;
	grade_documents -.-> generate;
	grade_documents -.-> rewrite_query;
	retrieve --> grade_documents;
	rewrite_query --> retrieve;
	abstain --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### **Step 7: Run it**
`ask` starts every question from a fresh state, and `thread_id` keeps the saved checkpoints separate.

In [27]:
import time

def ask(question, thread_id="demo"):
    start = time.time()
    result = agent.invoke(initial_state(question), {"configurable": {"thread_id": thread_id}})
    result["latency"] = time.time() - start
    return result

def try_agent(question, thread_id="demo"):
    r = ask(question, thread_id)
    print("Q:", question)
    print("status:", r["status"], "| reason:", r["abstain_reason"], f"| {r['latency']:.1f}s")
    print("path:", " -> ".join(r["path"]))
    print("cited:", get_citations(r["answer"]))
    print(r["answer"])
    print("-" * 70)

try_agent("What is the maximum normal working week?", "t1")

Q: What is the maximum normal working week?
status: answered | reason:  | 9.3s
path: check_scope -> retrieve -> grade(2/5) -> generate -> grounding(ok)
cited: ['s.60A']
Not more than forty-five hours in one week [s.60A].
----------------------------------------------------------------------


### **Test every route**
Read the **path** each time. It should match the route you expect.

In [28]:
try_agent("My employer makes me work 12 hours a day. Is this allowed?", "t2")       # normal answer
try_agent("can my boss make me work crazy long", "t3")                              # badly worded: may need a rewrite
try_agent("What are my rights as a worker in Sabah under this Act?", "t4")          # should stop at check_scope
try_agent("What is the latest 2027 income tax relief limit?", "t5")                 # should stop at check_scope
try_agent("What remedies can the Industrial Court award for unfair dismissal?", "t6")   # not in this Act: expect abstain

Q: My employer makes me work 12 hours a day. Is this allowed?
status: answered | reason:  | 88.6s
path: check_scope -> retrieve -> grade(3/5) -> generate -> grounding(fail) -> generate -> grounding(ok)
cited: ['s.60A', 's.60C']
- Not a shift worker: Under s.60A, an employee shall not be required to work more than eight hours in one day, and not more than forty-five hours in one week. There is a possible exception by agreement where the number of hours on some days less than eight may be exceeded, but no employee shall be required to work more than nine hours in one day or forty-five hours in one week [s.60A].

- Shift worker: Under s.60C, a shift worker may be required to work more than eight hours in one day or more than forty-five hours in a week, but the average number of hours worked over any period of three weeks shall not exceed forty-five per week, and this is subject to Director General approval (s.60C(1), with (1a) for approval and (1b) for revocation) [s.60C(1), (1a), (1b)].


### **Common problems and where to fix them**
- The grader rejects good sections -> loosen `grade_prompt`.
- The grounding check rejects correct answers -> make `grounding_prompt` less strict.
- The scope check blocks valid questions -> adjust `scope_prompt`.
- A node returns a key that is not in `graph_schema` -> check the spelling in the state and the node.

---
# **Part B: Evaluation (baseline vs agent)**

Both systems answer the **same questions** in `../eval/questions.csv`. We measure:
- **retrieval hit:** was an expected section retrieved?
- **citation ok:** did the answer cite an expected section, and only sections that were retrieved?
- **faithful:** is every claim supported? Judged by a **third model** (`judge_llm`), not the one that wrote the answer.
- **behaviour ok:** did it answer when it should, and abstain when it should?
- **latency**

Report **counts** (for example 17/20), not only percentages, because the question set is small.

In [29]:
def baseline_answer(question):
    start = time.time()
    docs = retrieve_docs(question)
    answer = generate_chain.invoke({"context": format_context(docs), "question": question, "feedback": ""}).content.strip()
    return {"answer": answer, "documents": docs, "abstained": INSUFFICIENT in answer, "latency": time.time() - start}

judge_chain = grounding_prompt | judge_llm.with_structured_output(grounding_schema)

In [30]:
import pandas as pd

questions = pd.read_csv("../eval/questions.csv")
print(len(questions), "questions")
questions.head()

23 questions


,id,category,question,reference_answer,expected_sections,expected_behaviour
0,1,direct,What is the maximum normal working week?,"45 hours in one week (also max 8 hours a day, ...",60A,answer
1,2,direct,How long is the paid maternity leave?,An eligible period of not less than 98 consecu...,37,answer
2,3,direct,How much notice must be given to terminate a c...,Not less than 4 weeks (under 2 years of servic...,12,answer
3,4,direct,How many days of paid sick leave is an employe...,Without hospitalisation: 14 / 18 / 22 days a y...,60F,answer
4,5,direct,How many days of annual leave does an employee...,12 days for every 12 months of continuous serv...,60E,answer


In [31]:
def secs(x):
    return {s.strip() for s in str(x).split(";") if s.strip() and s.strip() != "nan"}

def score(row, docs, answer, abstained):
    expected = secs(row["expected_sections"])
    retrieved = {d["section_no"] for d in docs}
    cited = {c[2:] for c in get_citations(answer)}          # "s.60A" -> "60A"
    should_abstain = row["expected_behaviour"] == "abstain"

    out = {"behaviour_ok": abstained == should_abstain,
           "retrieval_hit": None, "citation_ok": None, "faithful": None}

    if not should_abstain and expected:
        out["retrieval_hit"] = bool(expected & retrieved)
        out["citation_ok"] = bool(expected & cited) and cited <= (expected | retrieved)

    if not abstained and docs:
        verdict = judge_chain.invoke({"context": format_context(docs), "answer": answer})
        out["faithful"] = verdict.supported

    return out

In [32]:
import time

rows = []
for _, row in questions.iterrows():
    q = row["question"]
    for attempt in range(3):
        try:
            b = baseline_answer(q)
            a = ask(q, thread_id=f"eval-{row['id']}")
            a_abstained = a["status"] == "abstained"

            for name, docs, answer, abstained, latency in [
                ("baseline", b["documents"], b["answer"], b["abstained"], b["latency"]),
                ("agentic", a["documents"], a["answer"], a_abstained, a["latency"]),
            ]:
                s = score(row, docs, answer, abstained)
                rows.append({"id": row["id"], "category": row["category"], "system": name,
                             "abstained": abstained, "latency": latency, "answer": answer[:300], **s})
            break
        except Exception as e:
            print(f"Retrying id={row['id']} due to: {e}")
            time.sleep(5 * (attempt + 1))

    pd.DataFrame(rows).to_csv("../eval/results.csv", index=False)
    print("done", row["id"])

results = pd.DataFrame(rows)

done 1
done 2
done 3
done 4
done 5
Retrying id=6 due to: Provider returned error
done 6
Retrying id=7 due to: Provider returned error
done 7
done 8
Retrying id=9 due to: 1 validation error for grounding_schema
unsupported_claims
  Input should be a valid list [type=list_type, input_value='["An employer or an empl...orty-five hours total.]', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type
done 9
done 10
done 11
done 12
done 13
done 14
done 15
done 16
done 17
done 18
done 19
done 20
done 21
done 22
done 23


In [33]:
def frac(s):
    s = s.dropna()
    return f"{int(s.sum())}/{len(s)}"

summary = results.groupby("system").agg(
    retrieval_hit=("retrieval_hit", frac),
    citation_ok=("citation_ok", frac),
    faithful=("faithful", frac),
    behaviour_ok=("behaviour_ok", frac),
    avg_latency_s=("latency", "mean"),
).round(2)

summary

,retrieval_hit,citation_ok,faithful,behaviour_ok,avg_latency_s
system,,,,,
agentic,16/18,13/18,15/15,20/23,20.73
baseline,16/18,15/18,16/17,20/23,16.86


### **Read the failures, not only the summary**
Open `results.csv` and look at the rows where `behaviour_ok`, `retrieval_hit` or `citation_ok` is False.
For each failure write one line in `notes.md`: what went wrong and why. That is your best interview material.

In [34]:
results[(results["behaviour_ok"] == False) | (results["retrieval_hit"] == False) | (results["citation_ok"] == False)][["id", "category", "system", "abstained", "answer"]]

,id,category,system,abstained,answer
9,5,direct,agentic,True,I could not produce an answer that was fully s...
16,9,direct,baseline,True,INSUFFICIENT_EVIDENCE
17,9,direct,agentic,False,- The Director General may grant permission to...
25,13,multi_section,agentic,True,I could not find enough reliable evidence in t...
26,14,scenario,baseline,True,INSUFFICIENT_EVIDENCE
27,14,scenario,agentic,True,I could not find enough reliable evidence in t...
34,18,ambiguous,baseline,False,"Yes, if you are an employee and you work overt..."
35,18,ambiguous,agentic,False,Yes. The overtime rule generally applies to em...
36,19,unsupported,baseline,False,- Sabah is outside the scope of the Employment...
